[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/19_imbalanced_learning/02_resampling_and_smote.ipynb)

# Bài 02 — Resampling: undersampling, oversampling, SMOTE

**🎯 Mục tiêu**
- Random undersampling / oversampling và **SMOTE** — cài đặt from scratch.
- Đặt resampling **đúng chỗ**: chỉ trên train của mỗi fold (meta-estimator) — và hậu quả khi resample trước khi chia.
- Resampling làm **sai lệch xác suất** — và công thức hiệu chỉnh prior.
- Khi nào resampling giúp, khi nào không.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/19_imbalanced_learning

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import metrics as skm
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate, train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course.datasets import make_fraud_transactions
from ml_course.visualization import PALETTE, set_style

set_style()
rng = np.random.default_rng(0)

## 1. Ba cách resampling from scratch

In [ ]:
def random_undersample(X, y, ratio=1.0, seed=0):
    """Giữ mọi mẫu thiểu số (1), lấy ngẫu nhiên ratio × số thiểu số mẫu đa số (0)."""
    r = np.random.default_rng(seed)
    pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    keep = np.r_[pos, r.choice(neg, min(len(neg), int(ratio * len(pos))), replace=False)]
    return X[keep], y[keep]


def random_oversample(X, y, seed=0):
    """Lặp lại (có hoàn lại) mẫu thiểu số cho tới khi bằng lớp đa số."""
    r = np.random.default_rng(seed)
    pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    extra = r.choice(pos, len(neg) - len(pos), replace=True)
    keep = np.r_[neg, pos, extra]
    return X[keep], y[keep]


def smote(X, y, k=5, seed=0):
    """SMOTE: mẫu mới = x_i + u · (láng giềng thiểu số ngẫu nhiên − x_i), u ~ U(0, 1)."""
    r = np.random.default_rng(seed)
    X_pos = X[y == 1]
    n_new = int((y == 0).sum() - (y == 1).sum())
    nn = NearestNeighbors(n_neighbors=k + 1).fit(X_pos)
    _, neigh = nn.kneighbors(X_pos)                         # cột 0 là chính nó
    base = r.integers(0, len(X_pos), n_new)
    partner = neigh[base, r.integers(1, k + 1, n_new)]
    u = r.random((n_new, 1))
    synthetic = X_pos[base] + u * (X_pos[partner] - X_pos[base])
    return np.vstack([X, synthetic]), np.r_[y, np.ones(n_new, dtype=y.dtype)]


X2 = np.vstack([rng.normal([0, 0], 1.0, (500, 2)), rng.normal([2.2, 1.8], 0.5, (25, 2))])
y2 = np.r_[np.zeros(500, int), np.ones(25, int)]
fig, axes = plt.subplots(1, 4, figsize=(22, 4.8))
for ax, (name, (Xr, yr)) in zip(axes, [("gốc", (X2, y2)), ("undersampling", random_undersample(X2, y2)),
                                       ("oversampling (lặp lại)", random_oversample(X2, y2)), ("SMOTE", smote(X2, y2))]):
    ax.scatter(*Xr[yr == 0].T, s=8, alpha=0.4, color=PALETTE[0], label="đa số")
    ax.scatter(*Xr[yr == 1].T, s=8, alpha=0.6, color=PALETTE[1], label="thiểu số")
    ax.set_title(f"{name}: {int((yr == 0).sum())} vs {int((yr == 1).sum())}")
axes[0].legend()
plt.show()

- **Undersampling**: vứt bỏ dữ liệu đa số — nhanh nhưng mất thông tin.
- **Oversampling**: lặp lại mẫu thiểu số — dễ overfit vào các điểm lặp.
- **SMOTE**: tạo điểm mới trên **đoạn thẳng** giữa hai mẫu thiểu số gần nhau — "làm dày" vùng thiểu số; giả định vùng giữa hai điểm thiểu số cũng là thiểu số (có thể sai nếu lớp đan xen).

## 2. Resampling đúng chỗ: chỉ trên train của mỗi fold

scikit-learn `Pipeline` không hỗ trợ bước thay đổi số mẫu, nên ta viết một **meta-estimator**: resample trong `fit`, dự đoán bình thường. (Thư viện `imbalanced-learn` có `imblearn.pipeline.Pipeline` làm việc này.)

In [ ]:
class ResampledClassifier(ClassifierMixin, BaseEstimator):
    def __init__(self, estimator, sampler=None, seed=0):
        self.estimator, self.sampler, self.seed = estimator, sampler, seed

    def fit(self, X, y):
        X, y = np.asarray(X, float), np.asarray(y)
        if self.sampler is not None:
            X, y = self.sampler(X, y, seed=self.seed)
        self.estimator_ = clone(self.estimator).fit(X, y)
        self.classes_ = self.estimator_.classes_
        return self

    def predict_proba(self, X):
        return self.estimator_.predict_proba(np.asarray(X, float))

    def predict(self, X):
        return self.estimator_.predict(np.asarray(X, float))


fraud = make_fraud_transactions(n=40000, fraud_rate=0.01, seed=21)
features = ["amount_vnd", "hour", "distance_from_home_km", "is_foreign", "n_tx_last_24h", "card_age_days"]
X = fraud[features].astype(float).assign(amount_vnd=lambda d: np.log(d.amount_vnd)).to_numpy()
y = fraud["is_fraud"].to_numpy()
cv = StratifiedKFold(5, shuffle=True, random_state=0)
base = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
rows = []
for name, sampler in [("không resample", None), ("undersample 1:1", random_undersample), ("oversample", random_oversample), ("SMOTE", smote)]:
    res = cross_validate(ResampledClassifier(base, sampler), X, y, cv=cv, scoring=["roc_auc", "average_precision", "neg_log_loss"])
    rows.append({"cách": name, "ROC-AUC": res["test_roc_auc"].mean(), "PR-AUC": res["test_average_precision"].mean(),
                 "log-loss": -res["test_neg_log_loss"].mean()})
pd.DataFrame(rows).round(4)

Quan sát quan trọng:
- **ROC-AUC và PR-AUC gần như không đổi** — resampling không làm Logistic Regression **xếp hạng** tốt hơn.
- **Log-loss tệ hơn hẳn** — xác suất dự đoán bị thổi phồng: model học trên dữ liệu 50/50 nên tin rằng gian lận phổ biến như giao dịch bình thường.

Phần lớn "lợi ích" của resampling trong các bài viết là do so sánh ở **ngưỡng 0.5** — resampling chỉ đơn giản là dịch ngưỡng một cách gián tiếp (và tốn kém).

## 3. Hiệu chỉnh prior sau resampling

Nếu train trên dữ liệu có tỉ lệ positive $\pi_s$ (sau resample) nhưng thực tế là $\pi$, xác suất đúng là:

$$p_{\text{đúng}} = \frac{p_s\,\frac{\pi}{\pi_s}}{p_s\,\frac{\pi}{\pi_s} + (1 - p_s)\,\frac{1 - \pi}{1 - \pi_s}}$$

In [ ]:
def prior_correct(p_s, pi, pi_s):
    a = p_s * pi / pi_s
    b = (1 - p_s) * (1 - pi) / (1 - pi_s)
    return a / (a + b)


X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
under = ResampledClassifier(base, random_undersample).fit(X_tr, y_tr)
p_s = under.predict_proba(X_te)[:, 1]
p_fix = prior_correct(p_s, pi=y_tr.mean(), pi_s=0.5)
p_orig = clone(base).fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
for name, pp in [("không resample", p_orig), ("undersample (thô)", p_s), ("undersample + hiệu chỉnh prior", p_fix)]:
    print(f"{name:32s} xác suất trung bình = {pp.mean():.4f} (thực tế {y_te.mean():.4f}) | log-loss = {skm.log_loss(y_te, pp):.4f} | "
          f"ROC-AUC = {skm.roc_auc_score(y_te, pp):.4f}")

Hiệu chỉnh prior đưa xác suất về đúng thang (log-loss gần như bằng model không resample) mà **không đổi thứ hạng** (ROC-AUC giữ nguyên — phép biến đổi đơn điệu).

## 4. Bẫy: resample **trước** khi chia

Oversample toàn bộ dữ liệu rồi mới chia train/test → bản sao của cùng một giao dịch gian lận nằm ở cả train và test → model "nhớ" chúng → kết quả ảo:

In [ ]:
X_over, y_over = random_oversample(X, y)
Xa, Xb, ya, yb = train_test_split(X_over, y_over, test_size=0.3, random_state=0, stratify=y_over)
rf = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=1)
wrong = skm.recall_score(yb, rf.fit(Xa, ya).predict(Xb))
right = skm.recall_score(y_te, ResampledClassifier(rf, random_oversample).fit(X_tr, y_tr).predict(X_te))
print(f"oversample TRƯỚC khi chia: recall gian lận trên 'test' = {wrong:.3f}   ← ảo tưởng")
print(f"oversample CHỈ trên train: recall gian lận trên test thật = {right:.3f}")

Đánh giá đúng cho thấy sự thật phũ phàng: Random Forest học trên các **bản sao** gian lận gần như chỉ "thuộc lòng" chúng — ở ngưỡng 0.5 hầu như không bắt được gian lận mới. Oversampling không tạo ra thông tin mới.

## 5. Khi nào resampling có ích?

- Model **không hỗ trợ trọng số mẫu** và ta cần ranh giới ở vùng thiểu số (vài thuật toán cũ).
- **Undersampling để tăng tốc** khi lớp đa số khổng lồ (hàng trăm triệu mẫu) — kèm hiệu chỉnh prior.
- Ensemble nhiều lần undersampling khác nhau (EasyEnsemble, Balanced Random Forest) — giảm mất thông tin.
- Mọi trường hợp khác: thử **class weight** và **chọn ngưỡng** trước (bài 03) — đơn giản và không làm hỏng xác suất.

## 🧠 Kiểm tra nhanh

1. SMOTE tạo mẫu mới thế nào? Giả định gì?
2. Vì sao resampling thường không cải thiện ROC-AUC của Logistic Regression?
3. Undersample về 50/50 khi tỉ lệ thật là 1%: model dự đoán 0.6 cho một giao dịch. Xác suất đã hiệu chỉnh bằng bao nhiêu?
4. Vì sao oversample trước khi chia train/test cho kết quả ảo?

## 📌 Tóm tắt

- Under/oversampling, SMOTE: cài đặt đơn giản; phải nằm trong từng fold (meta-estimator/imblearn Pipeline).
- Resampling ≈ dịch ngưỡng gián tiếp; ít cải thiện xếp hạng; làm sai xác suất → hiệu chỉnh prior.
- Resample trước khi chia = rò rỉ.

## 📚 Tài liệu tham khảo

- Chawla et al. (2002) — *SMOTE: Synthetic Minority Over-sampling Technique*
- Elkan (2001) — *The Foundations of Cost-Sensitive Learning*
- Lemaître, Nogueira & Aridas (2017) — *Imbalanced-learn: A Python Toolbox*